# Bulk ML computation for all 14,803 catalog events

Iterate every row of `HypoInv/catalog_phasenet_plus_2010_2024_blastclean.csv` and compute an
event-level ML via the same chain validated in `01.Calculate_ML_with_full_response.ipynb`:

  raw counts → DISP via response deconvolution → Wood-Anderson sim → peak (mm)
              → distance attenuation (Heo et al. 2024 default) → median across station-channels.

Output: `catalog_phasenet_plus_2010_2024_blastclean_with_ml_heo.csv` — the original catalog plus
four columns conforming to the **`seismostats.Catalog` convention**:

  * `magnitude`       — median ML across station-channels (the column SeismoStats reads).
  * `magnitude_std`   — scatter (std) of the per-station MLs.
  * `n_used`          — number of station-channels contributing.
  * `mag_status`      — `"ok"` | `"no_dir"` | `"no_picks"` | `"error:<msg>"`.

The output CSV loads directly into `seismostats.Catalog.from_dict(df.to_dict())` for the
summary notebook `03.Magnitude_summary.ipynb`.

**Wall-clock budget.** ~1 s/event × 15.7k ÷ 4 threads ≈ **1 hour** end-to-end. The runner
is resumable: if interrupted (Ctrl-C, SIGKILL, OOM), re-running picks up at the first row
that has not yet been processed — the CSV is flushed to disk every 200 events.

## 1. Setup — inventory, catalog, attenuation choice

In [ ]:
import os, sys, glob, warnings
sys.path.insert(0, os.path.abspath("."))
import ml_pipeline as mlp
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")     # ObsPy "sensitivity differs by 5%" chatter, harmless

HERE = os.path.abspath(".")
HYP  = "/home/msseo/works/02.Ulsan_Fault_detection/KS_KG/HypoInv"

# --- attenuation -------------------------------------------------------
ATT_FN       = mlp.ml_heo2024          # Heo 2024 — GHBSN-calibrated, default for this catalog
RESTRICT_Z   = True                    # True → Z-only (Heo paper exact); False → all 3 comps

# --- paths -------------------------------------------------------------
CAT_IN   = f"{HYP}/catalog_phasenet_plus_2010_2024_blastclean.csv"
CAT_OUT  = f"{HERE}/catalog_phasenet_plus_2010_2024_blastclean_with_ml_heo.csv"
EVENT_ROOTS = (
    f"{HYP}/event_waveforms_blastclean",
    f"{HYP}/event_waveforms_ulsanfault",
)
MASTER_XML  = f"{HERE}/responses/master"
FETCHED_XML = f"{HERE}/responses/fetched"

# --- load --------------------------------------------------------------
inv = mlp.load_combined_inventory(MASTER_XML, FETCHED_XML)
cat = pd.read_csv(CAT_IN)
print(f"catalog : {len(cat):,} events  ({cat.time.iloc[0]} … {cat.time.iloc[-1]})")
print(f"inventory: {sum(len(n) for n in inv)} stations across {len(inv)} network(s)")
print(f"attenuation: {ATT_FN.__name__}, restrict_to_z={RESTRICT_Z}")
print(f"event_roots:")
for r in EVENT_ROOTS:
    n = len(os.listdir(r)) if os.path.isdir(r) else 0
    print(f"  {r}  →  {n:,} event dirs")

## 2. Coverage check — locate every catalog event's SAC dir

Before launching the long bulk run, confirm each catalog row resolves to a real directory
under one of the two `event_waveforms_*/` trees. This catches name-mismatch bugs cheaply.

In [ ]:
import re
def _stem(t):
    m = re.match(r"(\d{4})-(\d{2})-(\d{2})[ T](\d{2}):(\d{2}):(\d{2})", str(t))
    return "".join(m.groups()) if m else None

cat["event_stem"] = cat.time.map(_stem)
where = []
roots = list(EVENT_ROOTS)
existing = [set(os.listdir(r)) for r in roots]
for stem in cat.event_stem:
    hit = None
    for r, s in zip(roots, existing):
        if stem in s:
            hit = os.path.basename(r); break
    where.append(hit or "MISSING")
cov = pd.Series(where, name="root").value_counts()
print(cov.to_string())
print(f"\ntotal located: {(pd.Series(where) != 'MISSING').sum():,} / {len(cat):,}")

## 3. Trial run — 100 events for timing + sanity

Sample 100 events scattered across the catalog (every Nth row) and time them serially.
Display the first few augmented rows so you can sanity-check the magnitude scale before
launching the full bulk run.

In [ ]:
import time
sample = cat.iloc[::len(cat) // 100][:100].copy().reset_index(drop=True)
t0 = time.time()
trial = mlp.export_ml_catalog(sample, EVENT_ROOTS, inv,
                              attenuation_fn=ATT_FN, restrict_to_z=RESTRICT_Z,
                              workers=1, skip_existing=False, out_path=None, progress=True)
dt = time.time() - t0
print(f"\n100 events serial: {dt:.1f} s → {dt / 100:.2f} s/event")
print(f"  status counts:")
print(trial.mag_status.value_counts().to_string())
print(f"\nfirst 10 augmented rows:")
display(trial.head(10)[["time", "lat", "lon", "depth", "magnitude", "magnitude_std", "n_used", "mag_status"]].round(3))

est_serial_min = dt / 100 * len(cat) / 60
est_4w_min     = est_serial_min / 4
print(f"\nestimated full-catalog wall-clock:")
print(f"  serial  : {est_serial_min:6.1f} min")
print(f"  workers=4: {est_4w_min:6.1f} min  (recommended)")
print(f"  workers=8: {est_serial_min / 8:6.1f} min")

## 4. Bulk run — all 14,803 events

Threadpool over events with `workers=4` (good balance: each event reads a few hundred MB
of SAC, processes through obspy + numpy, then releases). The CSV is flushed every 200
events so a kill leaves a partial-but-valid file behind; re-running this cell resumes
from the first un-processed row (`skip_existing=True`).

**To restart from scratch**, delete `catalog_phasenet_plus_2010_2024_blastclean_with_ml_heo.csv`
before running, or pass `skip_existing=False`.

In [ ]:
aug = mlp.export_ml_catalog(cat, EVENT_ROOTS, inv,
                            attenuation_fn=ATT_FN, restrict_to_z=RESTRICT_Z,
                            workers=4, skip_existing=True, out_path=CAT_OUT,
                            progress=True)
print(f"\nfinal: {len(aug):,} rows → {CAT_OUT}")

## 5. Statistics — coverage, success rate, ML distribution

In [ ]:
aug = pd.read_csv(CAT_OUT)
print(f"events with magnitude: {aug.magnitude.notna().sum():,} / {len(aug):,} "
      f"({100*aug.magnitude.notna().mean():.1f}%)")
print(f"status counts:")
print(aug.mag_status.value_counts().to_string())
print(f"\nML distribution (events with status='ok'):")
ok = aug[aug.mag_status == "ok"]
print(ok.magnitude.describe().round(3).to_string())
print(f"\nn_used distribution:")
print(ok.n_used.describe().round(1).to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), dpi=110)
axes[0].hist(ok.magnitude, bins=np.arange(-1, 6.5, 0.1), color="#1f77b4", alpha=0.8)
axes[0].set(xlabel="Computed ML", ylabel="Event count", title="Magnitude distribution")
axes[0].grid(alpha=0.3)
axes[1].hist(ok.n_used, bins=np.arange(0, 200, 5), color="#ff7f0e", alpha=0.8)
axes[1].set(xlabel="n_used (station-channels)", ylabel="Event count",
            title=f"Station-channel coverage (median {ok.n_used.median():.0f})")
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Sanity-check the top-magnitude events

Confirm the bulk run reproduces the §5 benchmarks of `01.Calculate_ML_with_full_response.ipynb`
(M5.8 / M5.4 / M4.5 Gyeongju events). These should appear in the top of the magnitude-sorted
list at values close to KMA-published — within ±0.3 ML when using Heo 2024.

In [ ]:
top = ok.sort_values("magnitude", ascending=False).head(15)
display(top[["time", "lat", "lon", "depth", "magnitude", "magnitude_std", "n_used"]].round(3))

# Cross-check the three Gyeongju benchmark events
BENCH = {
    "2016-09-12 11:32:54": 5.8,   # mainshock
    "2016-09-12 10:44:32": 5.4,   # foreshock
    "2016-09-19 11:33:58": 4.5,   # large aftershock
}
print("\nbenchmark vs published KMA ML:")
for t, kma in BENCH.items():
    hit = ok[ok.time.str.startswith(t)]
    if not len(hit):
        print(f"  {t}  — NOT IN catalog with status=ok")
        continue
    r = hit.iloc[0]
    print(f"  {t}  KMA {kma}  →  computed {r.magnitude:.2f} "
          f"(±{r.magnitude_std:.2f}, n={int(r.n_used)})  "
          f"residual {r.magnitude - kma:+.2f}")

## 7. Provenance footer

The augmented catalog is committed as a build artifact (the bulk computation is the slow
step; re-running it should only be needed when the attenuation formula changes or when new
events are added). See `03.Magnitude_summary.ipynb` for the FMD + b-value + size-scaled
seismicity map built on top of this output.

In [ ]:
import datetime, json as _json
meta = dict(
    generated_at = datetime.datetime.utcnow().isoformat() + "Z",
    n_events     = int(len(aug)),
    n_with_ml    = int(aug.magnitude.notna().sum()),
    attenuation  = ATT_FN.__name__,
    restrict_to_z = RESTRICT_Z,
    event_roots  = list(EVENT_ROOTS),
    inventory    = dict(master=MASTER_XML, fetched=FETCHED_XML,
                        n_stations=sum(len(n) for n in inv)),
)
with open(CAT_OUT.replace(".csv", ".meta.json"), "w") as f:
    _json.dump(meta, f, indent=2)
print(_json.dumps(meta, indent=2))